In [ ]:
#Resizing

In [ ]:
from PIL import Image
import os

# Define source and destination folders
source_folder = 'E://datasets//HAM10000'
destination_folder = 'E://datasets//HAM10000//RESIZED'
desired_size = (256, 256)  # Replace with your desired (width, height)

# Create destination folder if it doesn't exist
os.makedirs(destination_folder, exist_ok=True)

# Loop through all files in source folder
for filename in os.listdir(source_folder):
    if filename.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tiff')):
        image_path = os.path.join(source_folder, filename)
        try:
            # Open and resize image
            with Image.open(image_path) as img:
                resized_img = img.resize(desired_size)
                # Save to destination folder
                resized_img.save(os.path.join(destination_folder, filename))
            # print(f"Processed: {filename}")
        except Exception as e:
            print(f"Failed to process {filename}: {e}")
print("Completed")

In [ ]:
#Artifact removal

In [ ]:
import os
import cv2
import matplotlib.pyplot as plt 
import PIL
import tensorflow as tf
import pathlib
import glob

input_url = 'E://datasets//HAM10000//RESIZED'
output_url = 'E://datasets//HAM10000//ARTIFACT_FREE'

for i in os.listdir(input_url):
    j = os.path.join(input_url,i)
    j = cv2.imread(j)
    grayscale=cv2.cvtColor(j,cv2.COLOR_RGB2GRAY)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(17,17))
    blackhat = cv2.morphologyEx(grayscale, cv2.MORPH_BLACKHAT, kernel)
    ret,thresh2 = cv2.threshold(blackhat,5,255,cv2.THRESH_BINARY)
    dst = cv2.inpaint(j,thresh2,1,cv2.INPAINT_TELEA)
    # plt.imshow(dst)
    output = os.path.join(output_url,i)
    cv2.imwrite(output,dst)
#     break

In [ ]:
#Train-test split (80-20)

In [ ]:
#auto splitting folder for classification and reult is in folder- given in root_folder
import os
import random
import shutil

def split_train_test_data(root_folder, train_ratio=0.8):
    for class_folder in os.listdir(root_folder):
        class_folder_path = os.path.join(root_folder, class_folder)
        if not os.path.isdir(class_folder_path):
            continue
        
        files = os.listdir(class_folder_path)
        random.shuffle(files)
        train_size = int(len(files) * train_ratio)
        train_files = files[:train_size]
        test_files = files[train_size:]
        
        for file_name in train_files:
            src_path = os.path.join(class_folder_path, file_name)
            dest_path = os.path.join(root_folder, 'train', class_folder, file_name)
            os.makedirs(os.path.dirname(dest_path), exist_ok=True)
            shutil.copy(src_path, dest_path)
        
        for file_name in test_files:
            src_path = os.path.join(class_folder_path, file_name)
            dest_path = os.path.join(root_folder, 'test', class_folder, file_name)
            os.makedirs(os.path.dirname(dest_path), exist_ok=True)
            shutil.copy(src_path, dest_path)

# Example usage
root_folder = "E://datasets//HAM10000//ARTIFACT_FREE"
split_train_test_data(root_folder, train_ratio=0.8)

In [ ]:
#Augmentation(only train set)

In [ ]:
!pip install Augmentor

In [ ]:
import Augmentor

p = Augmentor.Pipeline("E://datasets//HAM10000//ARTIFACT_FREE//SPLIT//train//akiec")
p.rotate(probability=0.7, max_left_rotation=10, max_right_rotation=10)
p.zoom(probability=0.3, min_factor=1.1, max_factor=1.6)
p.flip_left_right(probability=0.3)
p.flip_top_bottom(probability=0.3)
p.sample(5364)

In [ ]:
#Creating .npy files

In [ ]:
import os
import numpy as np
from PIL import Image

def load_images_and_labels(data_dir, image_size=(224, 224)):
    X = []
    y = []
    class_names = sorted(os.listdir(data_dir))
    class_to_idx = {cls_name: idx for idx, cls_name in enumerate(class_names)}

    for cls in class_names:
        cls_folder = os.path.join(data_dir, cls)
        if not os.path.isdir(cls_folder):
            continue
        for filename in os.listdir(cls_folder):
            if filename.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tiff')):
                img_path = os.path.join(cls_folder, filename)
                try:
                    img = Image.open(img_path).convert('RGB')
                    img = img.resize(image_size)
                    X.append(np.array(img))
                    y.append(class_to_idx[cls])
                except Exception as e:
                    print(f"Error reading {img_path}: {e}")
    return np.array(X), np.array(y), class_to_idx

# Paths
train_folder = 'E://datasets//HAM10000//ARTIFACT_FREE//SPLIT//train'
test_folder = 'E://datasets//HAM10000//ARTIFACT_FREE//SPLIT//test'

# Load data
X_train, y_train, class_map = load_images_and_labels(train_folder)
X_test, y_test, _ = load_images_and_labels(test_folder)

# Save to .npy
np.save('E://datasets//HAM10000//ARTIFACT_FREE//SPLIT//X_train.npy', X_train)
np.save('E://datasets//HAM10000//ARTIFACT_FREE//SPLIT//y_train.npy', y_train)
np.save('E://datasets//HAM10000//ARTIFACT_FREE//SPLIT//X_test.npy', X_test)
np.save('E://datasets//HAM10000//ARTIFACT_FREE//SPLIT//y_test.npy', y_test)

print(f"Saved {X_train.shape[0]} training images and {X_test.shape[0]} test images.")
print(f"Class mapping: {class_map}")
